# SXF — Isolated GPU Benchmark V2 (resumable Vision verification)

Select **Runtime → Change runtime type → T4 GPU**, then **Run all**. No paid GPU service is required. By default only Vision runs, avoiding a repeat of the already successful Qwen3 8B measurements.

Evidence is always review-only. The ZIP includes all successful and partial records even if Vision fails. Reusing earlier text results requires their original complete V2 JSON in OUTPUT, with provenance unchanged. Colab runtimes do not automatically retain previous sessions' files.

To intentionally repeat both models, set WORKLOAD to `all` in the benchmark cell. Never interpret GPU-wide usage as model-only VRAM.

Each attempt exports the benchmark exit code, original stdout and stderr, and exact source commit in the review-only ZIP. The notebook checks it fetched code supporting Vision-only, even when opened from a PR link.

**Current Vision-only target:** `qwen3-vl:4b-instruct`, a different model variant from prior `qwen3-vl:4b` thinking captures. Never combine their results or claim the older model was verified.


In [ ]:
import pathlib,subprocess,sys,shutil,importlib
ROOT=pathlib.Path("/content/sxf-ai")
OUTPUT=pathlib.Path("/content/sxf-isolated-gpu-v2")
OUTPUT.mkdir(parents=True,exist_ok=True)
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No NVIDIA GPU assigned in Colab. Choose T4 GPU; no benchmark was run.")
REPO_URL="https://github.com/rab2323s-creator/sxf-ai.git"
# main is preferred after PR merge; until then, fall back to the feature branch.
FALLBACK_REF="fix/isolated-vision-resumable-evidence-v3"
if not (ROOT/".git").is_dir():
    subprocess.run(["git","clone","--depth","1","--branch","main",REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"fetch","--depth","1","origin","main"],check=True)
    subprocess.run(["git","-C",str(ROOT),"checkout","--detach","FETCH_HEAD"],check=True)
def supports_vision_resume():
    script=ROOT/"scripts/measure_isolated_gpu_evidence.py"
    source=script.read_text() if script.is_file() else ""
    return 'parser.add_argument("--workload"' in source and 'def valid_completed_report(' in source
if not supports_vision_resume():
    print("main does not contain the Vision-only runner; fetching feature branch.",flush=True)
    subprocess.run(["git","-C",str(ROOT),"fetch","--depth","1","origin",FALLBACK_REF],check=True)
    subprocess.run(["git","-C",str(ROOT),"checkout","--detach","FETCH_HEAD"],check=True)
if not supports_vision_resume():
    raise RuntimeError("Wrong GitHub revision: Vision-only runner unavailable. No benchmark attempted.")
BENCHMARK_SHA=subprocess.check_output(["git","-C",str(ROOT),"rev-parse","HEAD"],text=True).strip()
print("Benchmark source commit:",BENCHMARK_SHA,flush=True)
sys.path.insert(0,str(ROOT/"scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
OLLAMA=ensure_colab_ollama.ensure_ollama_ready(OUTPUT)
print("GPU assigned and Ollama ready:",OLLAMA)

In [ ]:
import subprocess,sys,json,datetime
WORKLOAD="vision"  # "vision" (default), "text" or "all"
ATTEMPT=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
started=datetime.datetime.now(datetime.timezone.utc).isoformat()
command=[sys.executable,"-u","scripts/measure_isolated_gpu_evidence.py",
    "--output-dir",str(OUTPUT),"--ollama-bin",OLLAMA,
    "--context","4096","--repeats","3","--workload",WORKLOAD]
run=subprocess.run(command,cwd=ROOT,capture_output=True,text=True)
(OUTPUT/("benchmark-stdout-"+ATTEMPT+".log")).write_text(run.stdout)
(OUTPUT/("benchmark-stderr-"+ATTEMPT+".log")).write_text(run.stderr)
(OUTPUT/("benchmark-run-"+ATTEMPT+".json")).write_text(json.dumps({
    "kind":"sxf-benchmark-execution-diagnostics",
    "publicationStatus":"do-not-publish", "sourceCommit":BENCHMARK_SHA,
    "workload":WORKLOAD,"startedAtUtc":started,
    "finishedAtUtc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "exitCode":run.returncode
},indent=2)+"\n")
if run.stdout: print(run.stdout[-20000:],flush=True)
if run.stderr: print(run.stderr[-20000:],flush=True)
if run.returncode:
    print("WARNING: benchmark exited",run.returncode,"; stdout/stderr preserved in ZIP.",flush=True)
for path in sorted(OUTPUT.glob("*-isolated.json")):
    record=json.loads(path.read_text())
    print(record["modelTag"],"trial tok/s:",[x["tokensPerSecond"] for x in record["runRecords"] if not x["warmup"]])
    if record["workload"]=="vision":
        print("Vision answer (manual review required):",record["runRecords"][-1]["answer"])

In [ ]:
import pathlib,zipfile,json,hashlib,datetime
from google.colab import files
names=sorted(p.name for p in OUTPUT.glob("*-isolated.json"))
names+=sorted(p.name for p in OUTPUT.glob("*-partial-*.json"))
names+=sorted(p.name for p in OUTPUT.glob("*-setup-*.json"))
if (OUTPUT/"vision-square.png").is_file():names.append("vision-square.png")
if (OUTPUT/"ollama-server.log").is_file():names.append("ollama-server.log")
names+=sorted(p.name for p in OUTPUT.glob("benchmark-stdout-*.log"))
names+=sorted(p.name for p in OUTPUT.glob("benchmark-stderr-*.log"))
names+=sorted(p.name for p in OUTPUT.glob("benchmark-run-*.json"))
if not names:raise RuntimeError("No evidence files to export; see captured setup error above.")
print("Packaging evidence files:",names)
manifest={"kind":"sxf-isolated-gpu-evidence","reviewStatus":"manual-review-required" if any(n.endswith("-isolated.json") for n in names) else "incomplete-do-not-publish",
          "createdAtUtc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "files":[{"name":name,"sha256":hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest()} for name in names]}
(OUTPUT/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
archive=pathlib.Path("/content/sxf-isolated-gpu-evidence-v2-review.zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for name in names+["manifest.json"]:z.write(OUTPUT/name,name)
print("Evidence ready, unpublished:",archive)
files.download(str(archive))